# 11 | Land-Cover-Dependent LiDAR Waveform Response

This is a descriptive, within-granule study of native H5 SNR, Refh amplitude, noise-calibrated candidate signals, and waveform morphology. NAIP polygons define image-context labels; they are not calibrated pulse-footprint truth. Nominal H5 centers are used only for ROI membership, and image-to-CASALS registration and pulse-footprint size remain uncalibrated.

Every pulse center in an Included or Candidate ROI is retained. Included ROIs with a verified source-image date and image-footprint coverage form the primary sample; Candidate ROIs remain exploratory. No pulse is filtered by `good_snr`, SNR, amplitude, or waveform quality before calculating the candidate-signal denominator. All reported amplitudes are native digital counts, not calibrated reflectance.

The old whole-waveform `max(z) > 3` screen is not used as a detection-rate metric. Instead, a full-length noise-null maximum is calibrated from real standardized late-window background blocks with a block bootstrap that accounts for searching 2,728 bins. This is a provisional candidate-screen threshold; it does not estimate real-signal detection probability or produce an official CASALS return detection.

In [1]:
from pathlib import Path
import json, sys, math, gc
from datetime import datetime, date
import h5py, numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from matplotlib.path import Path as PolygonPath
from pyproj import Transformer
from scipy.stats import skew

ROOT=Path.cwd()
if not (ROOT/'data').is_dir(): ROOT=ROOT.parent
if not (ROOT/'data').is_dir(): raise FileNotFoundError('Run from the CASALS project root.')
sys.path.insert(0,str(ROOT))
from research.l1b_validation import roi_primary_eligibility,common_track_ids,aggregate_roi_track_responses,matched_track_differences,calibrate_noise_peak_threshold,candidate_signal_features,site_level_unit_counts
H5_DIR=ROOT/'data/raw/casals_l1b'; ROI_PATH=ROOT/'data/reference/landcover_rois/roi_manifest.geojson'
NAIP_ROOT=ROOT/'outputs/notebooks/06_waveform_gallery/method_improvement'
OUT_ROOT=ROOT/'outputs/notebooks/11_landcover_lidar_response/revised'; OUT_ROOT.mkdir(parents=True,exist_ok=True)
H5_PATHS=sorted(H5_DIR.glob('casals_l1b_202411*T*_001_02.h5'))
if len(H5_PATHS)!=2: raise FileNotFoundError(f'Expected two H5 granules, found {len(H5_PATHS)}')
ROI_MANIFEST=json.loads(ROI_PATH.read_text(encoding='utf-8'))
NAIP_DATE_BY_GRANULE=ROI_MANIFEST['metadata']['naip_acquisition_dates_by_granule']
NAIP_ASSETS={p.stem:{'tif':NAIP_ROOT/p.stem/'search_overview_naip.tif','json':NAIP_ROOT/p.stem/'search_overview_naip.json'} for p in H5_PATHS}
for stem,assets in NAIP_ASSETS.items():
 metadata=json.loads(assets['json'].read_text(encoding='utf-8'))
 dates=metadata.get('acquisition_dates',[])
 if len(dates)!=1 or dates[0]!=NAIP_DATE_BY_GRANULE[stem]: raise RuntimeError(f'NAIP acquisition date mismatch for {stem}: metadata={dates}, ROI manifest={NAIP_DATE_BY_GRANULE[stem]}')
 if not assets['tif'].is_file(): raise FileNotFoundError(assets['tif'])

print({'h5_inputs':[p.name for p in H5_PATHS],'naip_dates':NAIP_DATE_BY_GRANULE,'output_root':str(OUT_ROOT.relative_to(ROOT))})

{'h5_inputs': ['casals_l1b_20241112T165718_001_02.h5', 'casals_l1b_20241118T171757_001_02.h5'], 'naip_dates': {'casals_l1b_20241112T165718_001_02': '2023-06-02', 'casals_l1b_20241118T171757_001_02': '2022-06-15'}, 'output_root': 'outputs\\notebooks\\11_landcover_lidar_response\\revised'}


In [2]:
def audit_rois_and_collect_membership(h5_path,chunk_rows=250000):
 stem=h5_path.stem; assets=NAIP_ASSETS[stem]; meta=json.loads(assets['json'].read_text(encoding='utf-8')); image_date=meta['acquisition_dates'][0]
 with h5py.File(h5_path,'r') as h5:
  lon=np.asarray(h5['refh_longitude']); lat=np.asarray(h5['refh_latitude'])
  sweep=np.asarray(h5['sweep_num']); track=np.asarray(h5['track_num'])
  snr=np.asarray(h5['refh_snr']); good=np.asarray(h5['good_snr']); n=len(lon)
  h5_date=str(h5.attrs['start_utca'])[:10]
  flag_consistent=bool(np.array_equal(good,snr>=5.0))
 if len({len(lat),len(sweep),len(track),len(snr),len(good)})!=1: raise RuntimeError(f'Inconsistent H5 scalar lengths: {stem}')
 image_date_obj=datetime.strptime(image_date,'%Y-%m-%d').date(); h5_date_obj=datetime.strptime(h5_date,'%Y-%m-%d').date()
 roi_specs=[]; audit_rows=[]
 with rasterio.open(assets['tif']) as ds:
  bounds=ds.bounds; image_crs=ds.crs.to_string()
  transformer=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True)
  for feature in ROI_MANIFEST['features']:
   props=feature['properties']
   if props.get('granule_id')!=stem: continue
   coords=np.asarray(feature['geometry']['coordinates'][0],dtype=np.float64)
   px,py=transformer.transform(coords[:,0],coords[:,1])
   in_bounds=(np.asarray(px)>=bounds.left)&(np.asarray(px)<=bounds.right)&(np.asarray(py)>=bounds.bottom)&(np.asarray(py)<=bounds.top)
   footprint_fraction=float(in_bounds.mean())
   eligibility=roi_primary_eligibility(props,expected_granule_id=stem,observed_image_date=image_date)
   full_footprint=bool(in_bounds.all())
   primary=bool(eligibility['primary_eligible'] and full_footprint)
   tier='primary' if primary else ('exploratory' if props.get('analysis_status') in ('Candidate','Included') else 'excluded')
   roi_specs.append({'feature':feature,'props':props,'coords':coords,'primary':primary,'tier':tier})
   center=coords[:-1].mean(axis=0); cx,cy=transformer.transform(float(center[0]),float(center[1]))
   audit_rows.append({'granule_id':stem,'roi_id':props.get('roi_id'),'site_id':props.get('site_id'),'class_label':props.get('class_label'),'analysis_status':props.get('analysis_status'),'analysis_tier':tier,'primary_eligibility_reasons':'|'.join(eligibility['reasons']+([] if full_footprint else ['roi_not_fully_inside_naip_footprint'])),'naip_acquisition_date_metadata':image_date,'naip_date_in_roi_manifest':props.get('source_date'),'naip_date_match':bool(props.get('source_date')==image_date),'h5_acquisition_date':h5_date,'imagery_age_days_at_h5':int((h5_date_obj-image_date_obj).days),'naip_image_path':str(assets['tif'].relative_to(ROOT)),'naip_metadata_path':str(assets['json'].relative_to(ROOT)),'naip_crs':image_crs,'roi_vertices_inside_naip_fraction':footprint_fraction,'roi_fully_inside_naip_footprint':full_footprint,'image_alignment':'ROI vertices transformed from GeoJSON EPSG:4326 to NAIP raster CRS; H5 nominal centers tested in GeoJSON coordinates; no pulse-footprint registration calibration','confidence':props.get('confidence'),'label_source':props.get('label_source'),'evidence_source':props.get('evidence_source'),'temporal_change_risk':props.get('temporal_change_risk'),'temporal_limits':props.get('temporal_limits'),'boundary_mixed_surface_risk':props.get('boundary_risk'),'geometry_vertex_mean_x_epsg32618_m':float(cx),'geometry_vertex_mean_y_epsg32618_m':float(cy),'same_site_roi_count':sum(f['properties'].get('granule_id')==stem and f['properties'].get('site_id')==props.get('site_id') for f in ROI_MANIFEST['features'])})
 if not flag_consistent: raise RuntimeError('H5 good_snr is not exactly equal to refh_snr >= 5 for all records.')
 parts=[]; pulse_counts={spec['props']['roi_id']:0 for spec in roi_specs}; pulse_tracks={spec['props']['roi_id']:set() for spec in roi_specs}
 for start in range(0,n,chunk_rows):
  stop=min(start+chunk_rows,n); points=np.column_stack((lon[start:stop],lat[start:stop]))
  valid=np.all(np.isfinite(points),axis=1)
  for spec in roi_specs:
   props=spec['props']; mask=valid&PolygonPath(spec['coords']).contains_points(points)
   count=int(mask.sum()); pulse_counts[props['roi_id']]+=count
   if not count: continue
   pulse_tracks[props['roi_id']].update(np.unique(track[start:stop][mask]).astype(int).tolist())
   if spec['tier']=='excluded': continue
   ids=np.flatnonzero(mask)+start
   parts.append(pd.DataFrame({'granule_id':stem,'roi_id':props['roi_id'],'site_id':props.get('site_id'),'class_label':props.get('class_label'),'analysis_status':props.get('analysis_status'),'analysis_tier':spec['tier'],'pulse_index':ids.astype(np.int32),'sweep_num':sweep[ids].astype(np.int32),'track_num':track[ids].astype(np.int16),'longitude_deg':lon[ids],'latitude_deg':lat[ids]}))
 for row in audit_rows:
  rid=row['roi_id']; row['n_nominal_pulse_centers']=pulse_counts[rid]; row['n_tracks']=len(pulse_tracks[rid]); row['track_min']=min(pulse_tracks[rid]) if pulse_tracks[rid] else None; row['track_max']=max(pulse_tracks[rid]) if pulse_tracks[rid] else None
  row['primary_or_exploratory_analysis_rule']='Only Included ROIs with exact NAIP-date provenance and full image-footprint coverage are primary; Candidate/other Included ROIs are exploratory.'
 membership=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=['granule_id','roi_id','site_id','class_label','analysis_status','analysis_tier','pulse_index','sweep_num','track_num','longitude_deg','latitude_deg'])
 return pd.DataFrame(audit_rows),membership,{'n_h5_pulses':n,'h5_good_snr_equals_snr_ge_5':flag_consistent,'h5_acquisition_date':h5_date,'naip_date':image_date,'image_crs':image_crs}

ROI_AUDIT_FRAMES=[]; MEMBERSHIP_BY_GRANULE={}; GRANULE_AUDITS={}
for h5_path in H5_PATHS:
 audit,membership,granule_audit=audit_rois_and_collect_membership(h5_path)
 ROI_AUDIT_FRAMES.append(audit); MEMBERSHIP_BY_GRANULE[h5_path.stem]=membership; GRANULE_AUDITS[h5_path.stem]=granule_audit
 print(h5_path.stem,granule_audit,'ROI pulse memberships',len(membership))
ROI_AUDIT=pd.concat(ROI_AUDIT_FRAMES,ignore_index=True)
ROI_AUDIT.to_csv(OUT_ROOT/'roi_audit.csv',index=False)


casals_l1b_20241112T165718_001_02 {'n_h5_pulses': 3604480, 'h5_good_snr_equals_snr_ge_5': True, 'h5_acquisition_date': '2024-11-12', 'naip_date': '2023-06-02', 'image_crs': 'EPSG:32618'} ROI pulse memberships 81139


casals_l1b_20241118T171757_001_02 {'n_h5_pulses': 3604480, 'h5_good_snr_equals_snr_ge_5': True, 'h5_acquisition_date': '2024-11-18', 'naip_date': '2022-06-15', 'image_crs': 'EPSG:32618'} ROI pulse memberships 47288


In [3]:
def sample_real_background_windows(h5_path,window=(2600,2728),block_count=24,rows_per_block=256):
 """Take deterministic late-window noise samples spread across the full acquisition."""
 windows=[]
 with h5py.File(h5_path,'r') as h5:
  rx=h5['rx_waveform']; n=rx.shape[0]; lo,hi=window
  if hi>rx.shape[1] or hi-lo<16: raise ValueError(f'Invalid background window {window}')
  starts=np.linspace(0,n-rows_per_block,block_count,dtype=int)
  for start in starts:
   stop=min(start+rows_per_block,n)
   wave=np.asarray(rx[start:stop,lo:hi],dtype=np.float32)
   mean=np.asarray(h5['bg_mean'][start:stop],dtype=np.float32)
   std=np.asarray(h5['bg_std'][start:stop],dtype=np.float32)
   valid=np.isfinite(mean)&np.isfinite(std)&(std>0)&np.all(np.isfinite(wave),axis=1)
   if valid.any(): windows.append((wave[valid]-mean[valid,None])/std[valid,None])
 if not windows: raise RuntimeError(f'No valid background windows in {h5_path.name}')
 return np.concatenate(windows,axis=0).astype(np.float32,copy=False)


def background_stationarity_table(windows):
 rows=[]
 for segment in range(4):
  start=segment*(windows.shape[1]//4); stop=(segment+1)*(windows.shape[1]//4)
  part=windows[:,start:stop]
  means=np.mean(part,axis=1); stds=np.std(part,axis=1)
  centered=part-part.mean(axis=1,keepdims=True)
  denom=np.sqrt(np.sum(centered[:,:-1]**2,axis=1)*np.sum(centered[:,1:]**2,axis=1))
  lag1=np.divide(np.sum(centered[:,:-1]*centered[:,1:],axis=1),denom,out=np.zeros(len(part)),where=denom>0)
  rows.append({'tail_segment':f'{start}:{stop}','bin_count':stop-start,'window_count':len(part),'median_standardized_mean':float(np.median(means)),'median_standardized_sd':float(np.median(stds)),'p95_absolute_standardized_sample':float(np.percentile(np.abs(part),95)),'median_lag1_correlation':float(np.median(lag1))})
 table=pd.DataFrame(rows); sd=table['median_standardized_sd'].to_numpy(); ratio=float(sd.max()/sd.min()) if sd.min()>0 else None
 return table,ratio

NOISE_CALIBRATIONS={}; NOISE_WINDOWS={}; NOISE_SEGMENT_TABLES=[]
for h5_path in H5_PATHS:
 stem=h5_path.stem; windows=sample_real_background_windows(h5_path)
 segment_table,sd_ratio=background_stationarity_table(windows)
 threshold=calibrate_noise_peak_threshold(windows,waveform_length=2728,quantile=0.995,minimum_peak_distance=5,calibration_simulations=10000,validation_simulations=10000,seed=20261012+(12 if '20241112' in stem else 18))
 threshold.update({'granule_id':stem,'background_window_bins':'2600:2728','background_source':'standardized real RX tail windows across the entire granule using per-pulse H5 bg_mean/bg_std','tail_segment_sd_ratio_max_min':sd_ratio,'tail_sd_stationarity_flag':bool(sd_ratio is not None and sd_ratio<=1.5),'detection_probability_calibrated':False,'interpretation':'provisional full-length candidate-screen threshold under an exchangeable stationary-tail block-bootstrap null; synthetic-null false-alarm rate only'})
 NOISE_CALIBRATIONS[stem]=threshold; NOISE_WINDOWS[stem]=windows; NOISE_SEGMENT_TABLES.append(segment_table.assign(granule_id=stem))
 (OUT_ROOT/stem).mkdir(parents=True,exist_ok=True)
 (OUT_ROOT/stem/'noise_peak_calibration.json').write_text(json.dumps(threshold,indent=2,allow_nan=False),encoding='utf-8')
 segment_table.assign(granule_id=stem).to_csv(OUT_ROOT/stem/'noise_tail_stationarity.csv',index=False)
 print({k:threshold[k] for k in ['granule_id','real_background_window_count','threshold_peak_prominence_sigma','validation_false_alarm_rate','tail_segment_sd_ratio_max_min','tail_sd_stationarity_flag','detection_probability_calibrated']})
NOISE_SEGMENT_TABLE=pd.concat(NOISE_SEGMENT_TABLES,ignore_index=True)
NOISE_SEGMENT_TABLE.to_csv(OUT_ROOT/'noise_tail_stationarity.csv',index=False)
(OUT_ROOT/'noise_peak_calibration.json').write_text(json.dumps(NOISE_CALIBRATIONS,indent=2,allow_nan=False),encoding='utf-8')


{'granule_id': 'casals_l1b_20241112T165718_001_02', 'real_background_window_count': 6144, 'threshold_peak_prominence_sigma': 10.88476848602295, 'validation_false_alarm_rate': 0.0058, 'tail_segment_sd_ratio_max_min': 1.0098564597652688, 'tail_sd_stationarity_flag': True, 'detection_probability_calibrated': False}


{'granule_id': 'casals_l1b_20241118T171757_001_02', 'real_background_window_count': 6144, 'threshold_peak_prominence_sigma': 10.194930076599121, 'validation_false_alarm_rate': 0.0029, 'tail_segment_sd_ratio_max_min': 1.0095909915723602, 'tail_sd_stationarity_flag': True, 'detection_probability_calibrated': False}


2557

In [4]:
def waveform_candidate_features(waveform,bg_mean,bg_std,threshold_sigma):
 raw=np.asarray(waveform,dtype=np.float64).reshape(-1)
 detected=candidate_signal_features(raw,bg_mean,bg_std,prominence_threshold_sigma=threshold_sigma,minimum_width_bins=2.0,minimum_peak_distance=5)
 out={'waveform_status':'invalid_waveform' if detected['candidate_signal_status'].startswith('invalid_') else ('candidate' if detected['candidate_signal_detected'] else 'no_candidate'),'waveform_quality_reason':detected['candidate_signal_status'],'finite_waveform_fraction':detected['finite_fraction'],'candidate_signal_detected':bool(detected['candidate_signal_detected']),'candidate_peak_bin':detected['candidate_peak_bin'] if detected['candidate_signal_detected'] else None,'candidate_peak_prominence_sigma':detected['candidate_peak_prominence_sigma'] if detected['candidate_signal_detected'] else None,'maximum_interior_peak_prominence_sigma':detected['candidate_peak_prominence_sigma'],'candidate_peak_width_bins':detected['candidate_peak_width_bins'] if detected['candidate_signal_detected'] else None,'width_halfmax_bins':detected['candidate_peak_width_bins'] if detected['candidate_signal_detected'] else None,'candidate_local_peak_count':int(detected['candidate_local_peak_count'])}
 out.update({'dominant_support_energy_counts_bin':None,'signal_extent_bins':None,'asymmetry_skewness':None,'candidate_peak_excess_counts':None})
 if not detected['candidate_signal_detected']: return out
 peak=int(detected['candidate_peak_bin']); z=(raw-float(bg_mean))/float(bg_std)
 out['candidate_peak_excess_counts']=float(raw[peak]-float(bg_mean))
 # Morphology uses the contiguous 1-SD support around a threshold-passing peak.
 above=np.isfinite(z)&(z>=1.0); left=peak; right=peak+1
 while left>0 and above[left-1]: left-=1
 while right<len(z) and above[right]: right+=1
 bins=np.arange(left,right,dtype=np.float64); excess=np.maximum(raw[left:right]-float(bg_mean),0.0)
 energy=float(excess.sum()); out['dominant_support_energy_counts_bin']=energy; out['signal_extent_bins']=int(right-left)
 if energy>0 and len(bins)>=3:
  center=float(np.dot(bins,excess)/energy); var=float(np.dot((bins-center)**2,excess)/energy)
  out['asymmetry_skewness']=float(np.dot((bins-center)**3,excess)/energy/(var**1.5)) if var>0 else None
 return out


def extract_roi_pulse_features(h5_path,membership):
 stem=h5_path.stem; pulse_ids=np.unique(membership['pulse_index'].to_numpy(dtype=np.int32))
 if not pulse_ids.size: raise RuntimeError(f'No candidate or included ROI pulse centers for {stem}.')
 threshold=float(NOISE_CALIBRATIONS[stem]['threshold_peak_prominence_sigma']); rows=[]
 with h5py.File(h5_path,'r') as h5:
  rx=h5['rx_waveform']; n=rx.shape[0]; row_chunk=int(rx.chunks[0]) if rx.chunks else 512
  scalar_names=('refh','refh_amp','refh_snr','good_snr','bg_mean','bg_std','sweep_num','track_num')
  for start in range(0,n,row_chunk):
   stop=min(start+row_chunk,n); lo=int(np.searchsorted(pulse_ids,start,'left')); hi=int(np.searchsorted(pulse_ids,stop,'left'))
   if lo==hi: continue
   ids=pulse_ids[lo:hi]
   wave_block=np.asarray(rx[start:stop,:])
   scalar={name:np.asarray(h5[name][start:stop]) for name in scalar_names}
   for pulse_id in ids:
    j=int(pulse_id-start); feature=waveform_candidate_features(wave_block[j],float(scalar['bg_mean'][j]),float(scalar['bg_std'][j]),threshold)
    rows.append({'granule_id':stem,'pulse_index':int(pulse_id),'sweep_num':int(scalar['sweep_num'][j]),'track_num':int(scalar['track_num'][j]),'refh_m':float(scalar['refh'][j]) if np.isfinite(scalar['refh'][j]) else None,'refh_amp_counts':int(scalar['refh_amp'][j]),'refh_snr':float(scalar['refh_snr'][j]) if np.isfinite(scalar['refh_snr'][j]) else None,'good_snr':bool(scalar['good_snr'][j]),'bg_mean_counts':float(scalar['bg_mean'][j]) if np.isfinite(scalar['bg_mean'][j]) else None,'bg_std_counts':float(scalar['bg_std'][j]) if np.isfinite(scalar['bg_std'][j]) else None,'noise_calibrated_prominence_threshold_sigma':threshold,**feature})
 pulse_features=pd.DataFrame(rows)
 if len(pulse_features)!=len(pulse_ids): raise RuntimeError(f'Waveform extraction lost ROI pulse records for {stem}.')
 per_roi=membership.merge(pulse_features,on=['granule_id','pulse_index','sweep_num','track_num'],how='left',validate='many_to_one')
 if per_roi['waveform_status'].isna().any(): raise RuntimeError('Some ROI pulse memberships did not receive a waveform record.')
 per_roi['candidate_signal_detected']=per_roi['candidate_signal_detected'].fillna(False).astype(bool)
 return pulse_features,per_roi

ROI_PULSE_FRAMES=[]; PULSE_FEATURES_BY_GRANULE={}
for h5_path in H5_PATHS:
 stem=h5_path.stem; membership=MEMBERSHIP_BY_GRANULE[stem]
 pulse_features,per_roi=extract_roi_pulse_features(h5_path,membership)
 PULSE_FEATURES_BY_GRANULE[stem]=pulse_features
 per_roi.to_csv(OUT_ROOT/stem/'roi_pulse_features.csv.gz',index=False,compression='gzip')
 ROI_PULSE_FRAMES.append(per_roi)
 print({'granule':stem,'unique_roi_pulses':len(pulse_features),'roi_pulse_rows':len(per_roi),'invalid_waveforms':int((per_roi.waveform_status=='invalid_waveform').sum()),'candidate_signals':int(per_roi.candidate_signal_detected.sum()),'all_roi_pulses_retained':True})
 del pulse_features,per_roi
 gc.collect()
ROI_PULSE_FEATURES=pd.concat(ROI_PULSE_FRAMES,ignore_index=True)


{'granule': 'casals_l1b_20241112T165718_001_02', 'unique_roi_pulses': 81139, 'roi_pulse_rows': 81139, 'invalid_waveforms': 0, 'candidate_signals': 12478, 'all_roi_pulses_retained': True}


{'granule': 'casals_l1b_20241118T171757_001_02', 'unique_roi_pulses': 47288, 'roi_pulse_rows': 47288, 'invalid_waveforms': 0, 'candidate_signals': 39007, 'all_roi_pulses_retained': True}


In [5]:
def json_safe_n11(value):
 if isinstance(value,dict): return {str(k):json_safe_n11(v) for k,v in value.items()}
 if isinstance(value,(list,tuple)): return [json_safe_n11(v) for v in value]
 if isinstance(value,np.ndarray): return json_safe_n11(value.tolist())
 if isinstance(value,np.bool_): return bool(value)
 if isinstance(value,np.integer): return int(value)
 if isinstance(value,(np.floating,float)): return float(value) if np.isfinite(value) else None
 if value is pd.NA or value is pd.NaT: return None
 return value

ROI_TRACK_ROWS=[]
for stem,frame in zip([p.stem for p in H5_PATHS],ROI_PULSE_FRAMES):
 ROI_TRACK_ROWS.extend(aggregate_roi_track_responses(frame))
ROI_TRACK_SUMMARY=pd.DataFrame(ROI_TRACK_ROWS)
track_timing=(ROI_PULSE_FEATURES.groupby(['granule_id','roi_id','site_id','class_label','track_num'],as_index=False)
 .agg(sweep_min=('sweep_num','min'),sweep_median=('sweep_num','median'),sweep_max=('sweep_num','max')))
ROI_TRACK_SUMMARY=ROI_TRACK_SUMMARY.merge(track_timing,on=['granule_id','roi_id','site_id','class_label','track_num'],how='left',validate='one_to_one')
ROI_TRACK_SUMMARY.to_csv(OUT_ROOT/'roi_track_response_summary.csv',index=False)

ROI_RESPONSE_ROWS=[]
for keys,g in ROI_PULSE_FEATURES.groupby(['granule_id','roi_id','site_id','class_label','analysis_status','analysis_tier'],sort=True,dropna=False):
 stem,rid,sid,cls,status,tier=keys; detected=g['candidate_signal_detected'].fillna(False).astype(bool); valid=g['waveform_status'].isin(['candidate','no_candidate'])
 row={'granule_id':stem,'roi_id':rid,'site_id':sid,'class_label':cls,'analysis_status':status,'analysis_tier':tier,'n_roi_center_pulses':int(len(g)),'n_valid_waveforms':int(valid.sum()),'n_invalid_waveforms':int((~valid).sum()),'n_candidate_signals':int(detected.sum()),'candidate_signal_rate_all_roi_pulses':float(detected.mean()),'n_no_candidate_valid_waveforms':int((valid&~detected).sum()),'h5_good_snr_true_n':int(g['good_snr'].sum()),'h5_good_snr_false_n':int((~g['good_snr']).sum()),'refh_snr_ge_3_n':int((g['refh_snr']>=3).sum()),'refh_snr_ge_5_n':int((g['refh_snr']>=5).sum()),'track_count':int(g['track_num'].nunique()),'pulse_denominator_rule':'all ROI pulse centers; no SNR, good_snr, amplitude, or waveform-quality filtering'}
 for col in ['refh_snr','refh_amp_counts']:
  vals=pd.to_numeric(g[col],errors='coerce').dropna().to_numpy()
  row[f'{col}_median']=float(np.median(vals)) if vals.size else None; row[f'{col}_p10']=float(np.percentile(vals,10)) if vals.size else None; row[f'{col}_p90']=float(np.percentile(vals,90)) if vals.size else None
  row[f'{col}_n']=int(vals.size)
 for col in ['width_halfmax_bins','dominant_support_energy_counts_bin','asymmetry_skewness']:
  vals=pd.to_numeric(g.loc[detected,col],errors='coerce').dropna().to_numpy()
  row[f'{col}_candidate_median']=float(np.median(vals)) if vals.size else None; row[f'{col}_candidate_p10']=float(np.percentile(vals,10)) if vals.size else None; row[f'{col}_candidate_p90']=float(np.percentile(vals,90)) if vals.size else None; row[f'{col}_candidate_n']=int(vals.size)
 row['refh_amp_negative_n']=int((g['refh_amp_counts']<0).sum())
 ROI_RESPONSE_ROWS.append(row)
ROI_RESPONSE=pd.DataFrame(ROI_RESPONSE_ROWS)
ROI_RESPONSE.to_csv(OUT_ROOT/'roi_response_summary.csv',index=False)

# Pair every cross-class ROI only within the same granule; a missing shared track is explicitly not comparable.
MATCHED_ROWS=[]; COMPARISON_SUMMARY=[]
metric_names=['refh_snr','refh_amp_counts','candidate_signal_rate','width_halfmax_bins','dominant_support_energy_counts_bin','asymmetry_skewness']
for stem in [p.stem for p in H5_PATHS]:
 active=ROI_AUDIT[(ROI_AUDIT.granule_id==stem)&(ROI_AUDIT.analysis_tier.isin(['primary','exploratory']))&(ROI_AUDIT.n_nominal_pulse_centers>0)]
 roi_records=active.to_dict('records')
 by_roi={rid:part.set_index('track_num',drop=False).to_dict('index') for rid,part in ROI_TRACK_SUMMARY[ROI_TRACK_SUMMARY.granule_id==stem].groupby('roi_id')}
 for i,audit_a in enumerate(roi_records):
  for audit_b in roi_records[i+1:]:
   if audit_a['class_label']==audit_b['class_label']: continue
   ra,rb=audit_a['roi_id'],audit_b['roi_id']; left=by_roi.get(ra,{}); right=by_roi.get(rb,{})
   tracks=common_track_ids(left.keys(),right.keys())
   distance=float(np.hypot(audit_a['geometry_vertex_mean_x_epsg32618_m']-audit_b['geometry_vertex_mean_x_epsg32618_m'],audit_a['geometry_vertex_mean_y_epsg32618_m']-audit_b['geometry_vertex_mean_y_epsg32618_m']))
   comp_tier='primary_pair' if audit_a['analysis_tier']=='primary' and audit_b['analysis_tier']=='primary' else 'exploratory_pair'
   if not tracks:
    MATCHED_ROWS.append({'granule_id':stem,'roi_a':ra,'roi_b':rb,'site_a':audit_a['site_id'],'site_b':audit_b['site_id'],'class_a':audit_a['class_label'],'class_b':audit_b['class_label'],'analysis_tier':comp_tier,'same_independent_site':audit_a['site_id']==audit_b['site_id'],'roi_vertex_mean_spacing_m':distance,'common_track_n':0,'comparability':'not_comparable_no_common_track','metric':'all','track_num':None,'difference_a_minus_b':None})
    COMPARISON_SUMMARY.append({'granule_id':stem,'roi_a':ra,'roi_b':rb,'class_a':audit_a['class_label'],'class_b':audit_b['class_label'],'analysis_tier':comp_tier,'same_independent_site':audit_a['site_id']==audit_b['site_id'],'roi_vertex_mean_spacing_m':distance,'common_track_n':0,'comparability':'not_comparable_no_common_track'})
    continue
   comparison_rows=[]
   for tr in tracks:
    left_row=left[tr]; right_row=right[tr]
    sweep_gap=float(abs(left_row['sweep_median']-right_row['sweep_median']))
    for metric in metric_names:
     if metric=='candidate_signal_rate': va,vb=left_row.get(metric),right_row.get(metric)
     else: va,vb=left_row.get(f'{metric}_median'),right_row.get(f'{metric}_median')
     if va is None or vb is None or not np.isfinite(va) or not np.isfinite(vb): continue
     record={'granule_id':stem,'roi_a':ra,'roi_b':rb,'site_a':audit_a['site_id'],'site_b':audit_b['site_id'],'class_a':audit_a['class_label'],'class_b':audit_b['class_label'],'analysis_tier':comp_tier,'same_independent_site':audit_a['site_id']==audit_b['site_id'],'roi_vertex_mean_spacing_m':distance,'common_track_n':len(tracks),'comparability':'common_track_descriptive_pair','metric':metric,'track_num':int(tr),'roi_a_median':float(va),'roi_b_median':float(vb),'difference_a_minus_b':float(va)-float(vb),'roi_a_pulse_n':int(left_row['n_pulses']),'roi_b_pulse_n':int(right_row['n_pulses']),'roi_a_sweep_median':float(left_row['sweep_median']),'roi_b_sweep_median':float(right_row['sweep_median']),'median_sweep_separation':sweep_gap}
     MATCHED_ROWS.append(record); comparison_rows.append(record)
   for metric in metric_names:
    vals=[r['difference_a_minus_b'] for r in comparison_rows if r['metric']==metric]
    if vals:
     COMPARISON_SUMMARY.append({'granule_id':stem,'roi_a':ra,'roi_b':rb,'class_a':audit_a['class_label'],'class_b':audit_b['class_label'],'analysis_tier':comp_tier,'same_independent_site':audit_a['site_id']==audit_b['site_id'],'roi_vertex_mean_spacing_m':distance,'common_track_n':len(tracks),'comparability':'common_track_descriptive_pair','metric':metric,'median_difference_across_tracks':float(np.median(vals)),'positive_track_fraction':float(np.mean(np.asarray(vals)>0)),'negative_track_fraction':float(np.mean(np.asarray(vals)<0))})
MATCHED_TRACK_COMPARISONS=pd.DataFrame(MATCHED_ROWS); MATCHED_TRACK_COMPARISONS.to_csv(OUT_ROOT/'matched_track_comparisons.csv',index=False)
MATCHED_TRACK_PAIR_SUMMARY=pd.DataFrame(COMPARISON_SUMMARY); MATCHED_TRACK_PAIR_SUMMARY.to_csv(OUT_ROOT/'matched_track_pair_summary.csv',index=False)

# Site is the independent unit. ROIs sharing a site ID are de-duplicated by pulse before aggregation.
site_pulses=ROI_PULSE_FEATURES.drop_duplicates(['granule_id','site_id','pulse_index']).copy()
SITE_ROWS=[]
SITE_ROI_COUNTS=ROI_PULSE_FEATURES.groupby(['granule_id','site_id','class_label','analysis_tier']).roi_id.nunique().to_dict()
for keys,g in site_pulses.groupby(['granule_id','site_id','class_label','analysis_tier'],sort=True,dropna=False):
 stem,sid,cls,tier=keys; detected=g.candidate_signal_detected.fillna(False).astype(bool); valid=g.waveform_status.isin(['candidate','no_candidate'])
 row={'granule_id':stem,'site_id':sid,'class_label':cls,'analysis_tier':tier,'n_rois':int(SITE_ROI_COUNTS.get((stem,sid,cls,tier),0)),'n_pulses_unique_within_site':int(len(g)),'n_valid_waveforms':int(valid.sum()),'candidate_signal_n':int(detected.sum()),'candidate_signal_rate_all_site_pulses':float(detected.mean()),'refh_snr_median':float(g.refh_snr.median()),'refh_snr_p10':float(g.refh_snr.quantile(.1)),'refh_snr_p90':float(g.refh_snr.quantile(.9)),'refh_amp_counts_median':float(g.refh_amp_counts.median()),'refh_amp_counts_p10':float(g.refh_amp_counts.quantile(.1)),'refh_amp_counts_p90':float(g.refh_amp_counts.quantile(.9)),'candidate_width_median_bins':float(g.loc[detected,'width_halfmax_bins'].median()) if detected.any() else None,'candidate_energy_median_counts_bin':float(g.loc[detected,'dominant_support_energy_counts_bin'].median()) if detected.any() else None,'candidate_asymmetry_median':float(g.loc[detected,'asymmetry_skewness'].median()) if detected.any() else None}
 SITE_ROWS.append(row)
SITE_RESPONSE=pd.DataFrame(SITE_ROWS); SITE_RESPONSE.to_csv(OUT_ROOT/'site_level_response_summary.csv',index=False)
SITE_VARIABILITY=[]
for (stem,tier,cls),g in SITE_RESPONSE.groupby(['granule_id','analysis_tier','class_label'],sort=True):
 for metric in ['refh_snr_median','refh_amp_counts_median','candidate_signal_rate_all_site_pulses','candidate_width_median_bins','candidate_energy_median_counts_bin']:
  vals=pd.to_numeric(g[metric],errors='coerce').dropna().to_numpy()
  SITE_VARIABILITY.append({'granule_id':stem,'analysis_tier':tier,'class_label':cls,'metric':metric,'independent_site_n':int(g.site_id.nunique()),'median_across_sites':float(np.median(vals)) if vals.size else None,'between_site_p10':float(np.percentile(vals,10)) if vals.size else None,'between_site_p90':float(np.percentile(vals,90)) if vals.size else None,'between_site_range':float(np.ptp(vals)) if vals.size>1 else None,'site_ids':'|'.join(sorted(g.site_id.astype(str).unique()))})
SITE_VARIABILITY=pd.DataFrame(SITE_VARIABILITY); SITE_VARIABILITY.to_csv(OUT_ROOT/'site_level_variability.csv',index=False)
SITE_UNIT_COUNTS={}
for stem in [p.stem for p in H5_PATHS]:
 part=SITE_RESPONSE[(SITE_RESPONSE.granule_id==stem)&(SITE_RESPONSE.analysis_tier=='primary')][['site_id','class_label']].drop_duplicates()
 SITE_UNIT_COUNTS[stem]=site_level_unit_counts(part)

SUMMARY={'study_name':'Land-Cover-Dependent LiDAR Waveform Response','interpretation':'Within-granule descriptive H5 waveform response; no calibrated reflectance, causal land-cover effect, or official CASALS return detection is inferred.','primary_roi_counts_by_granule':ROI_AUDIT[ROI_AUDIT.analysis_tier=='primary'].groupby('granule_id').size().to_dict(),'exploratory_roi_counts_by_granule':ROI_AUDIT[ROI_AUDIT.analysis_tier=='exploratory'].groupby('granule_id').size().to_dict(),'roi_audit':ROI_AUDIT.to_dict('records'),'site_unit_counts':SITE_UNIT_COUNTS,'site_level_response':SITE_RESPONSE.to_dict('records'),'site_level_variability':SITE_VARIABILITY.to_dict('records'),'noise_peak_calibration':NOISE_CALIBRATIONS,'matched_track_comparisons':MATCHED_TRACK_PAIR_SUMMARY.to_dict('records'),'good_snr_semantics':'For every complete H5 granule, good_snr is exactly refh_snr >= 5; these fields are not independent evidence sources.','denominator_rule':'All nominal H5 pulse centers inside each active Included or Candidate ROI, including invalid waveforms and no-candidate records.','amplitude_units':'Native H5 Refh amplitude in digital counts; not spectral or calibrated reflectance.'}
(OUT_ROOT/'landcover_response_summary.json').write_text(json.dumps(json_safe_n11(SUMMARY),indent=2,allow_nan=False),encoding='utf-8')
print({'roi_responses':len(ROI_RESPONSE),'roi_track_rows':len(ROI_TRACK_SUMMARY),'matched_track_rows':len(MATCHED_TRACK_COMPARISONS),'primary_site_counts':SITE_UNIT_COUNTS,'summary_path':str((OUT_ROOT/'landcover_response_summary.json').relative_to(ROOT))})


{'roi_responses': 11, 'roi_track_rows': 1714, 'matched_track_rows': 12471, 'primary_site_counts': {'casals_l1b_20241112T165718_001_02': {'open_field': {'n_independent_sites': 2, 'site_ids': ['nov12_east_cropland_01', 'nov12_southeast_cropland_02']}, 'tree_canopy': {'n_independent_sites': 1, 'site_ids': ['nov12_south_forest_stand_01']}}, 'casals_l1b_20241118T171757_001_02': {'open_field': {'n_independent_sites': 2, 'site_ids': ['field_site_01', 'field_site_02']}, 'road_surface': {'n_independent_sites': 1, 'site_ids': ['farm_road_segment_01']}, 'tree_canopy': {'n_independent_sites': 2, 'site_ids': ['nov18_northwest_woodlot_01', 'south_central_forest_patch']}}}, 'summary_path': 'outputs\\notebooks\\11_landcover_lidar_response\\revised\\landcover_response_summary.json'}


In [6]:

def save_figure(fig,directory,filename):
 fig.savefig(directory/filename,dpi=180,bbox_inches='tight'); plt.close(fig)

# Conservative same-track core: intersect track IDs across every Included primary ROI in a granule.
GLOBAL_COMMON_TRACKS={}; GLOBAL_PRIMARY_ROIS={}; GLOBAL_TRACK_ROWS=[]
for stem in [p.stem for p in H5_PATHS]:
 core=ROI_AUDIT[(ROI_AUDIT.granule_id==stem)&(ROI_AUDIT.analysis_tier=='primary')&(ROI_AUDIT.n_nominal_pulse_centers>0)].roi_id.astype(str).tolist()
 GLOBAL_PRIMARY_ROIS[stem]=core
 track_sets=[set(ROI_TRACK_SUMMARY.loc[(ROI_TRACK_SUMMARY.granule_id==stem)&(ROI_TRACK_SUMMARY.roi_id==rid),'track_num'].astype(int)) for rid in core]
 shared=sorted(set.intersection(*track_sets)) if track_sets else []
 GLOBAL_COMMON_TRACKS[stem]=shared
 for rid in core:
  part=ROI_TRACK_SUMMARY[(ROI_TRACK_SUMMARY.granule_id==stem)&(ROI_TRACK_SUMMARY.roi_id==rid)&(ROI_TRACK_SUMMARY.track_num.astype(int).isin(shared))]
  for row in part.to_dict('records'):
   GLOBAL_TRACK_ROWS.append({key:row.get(key) for key in ['granule_id','roi_id','site_id','class_label','analysis_tier','track_num','n_pulses','n_valid_waveforms','candidate_signal_rate','refh_snr_median','refh_amp_counts_median','width_halfmax_bins_median','dominant_support_energy_counts_bin_median','asymmetry_skewness_median','sweep_min','sweep_median','sweep_max']})
 print({'granule':stem,'primary_roi_n':len(core),'all_primary_roi_common_track_n':len(shared),'track_min':min(shared) if shared else None,'track_max':max(shared) if shared else None})
ALL_PRIMARY_ROI_COMMON_TRACK_RESPONSE=pd.DataFrame(GLOBAL_TRACK_ROWS)
ALL_PRIMARY_ROI_COMMON_TRACK_RESPONSE.to_csv(OUT_ROOT/'all_primary_roi_common_track_response_summary.csv',index=False)

# Image-only ROI review board; outlines are shown in the NAIP raster CRS.
for h5_path in H5_PATHS:
 stem=h5_path.stem; assets=NAIP_ASSETS[stem]; audit_part=ROI_AUDIT[ROI_AUDIT.granule_id==stem].set_index('roi_id')
 with rasterio.open(assets['tif']) as ds:
  if ds.count>=3: rgb=np.moveaxis(ds.read([1,2,3]),0,2)
  else: rgb=np.moveaxis(ds.read(),0,2)
  bounds=ds.bounds; transformer=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True)
  fig,ax=plt.subplots(figsize=(12,8),constrained_layout=True)
  ax.imshow(np.flipud(rgb),extent=(bounds.left,bounds.right,bounds.bottom,bounds.top),origin='lower')
  for feature in ROI_MANIFEST['features']:
   props=feature['properties']
   if props.get('granule_id')!=stem: continue
   coords=np.asarray(feature['geometry']['coordinates'][0],dtype=float); px,py=transformer.transform(coords[:,0],coords[:,1])
   roi_id=props['roi_id']; tier=audit_part.loc[roi_id,'analysis_tier']
   color={'primary':'#23e0a3','exploratory':'#ffca55','excluded':'#eeeeee'}[tier]
   ax.plot(px,py,color=color,lw=2,label=tier if tier not in [line.get_label() for line in ax.lines] else None)
   label_anchor=(float(np.mean(px)),float(np.mean(py)))
   label_offsets={'tree_canopy_01':(-72,-12),'tree_canopy_02':(72,12)}
   if roi_id in label_offsets:
    ax.annotate(roi_id,label_anchor,xytext=label_offsets[roi_id],textcoords='offset points',fontsize=8,color='white',ha='center',va='center',bbox={'facecolor':'black','alpha':0.55,'pad':2},arrowprops={'arrowstyle':'-','color':color,'lw':0.9,'alpha':0.8})
   else:
    ax.text(*label_anchor,roi_id,fontsize=8,color='white',ha='center',va='center',bbox={'facecolor':'black','alpha':0.55,'pad':2})
  date_value=GRANULE_AUDITS[stem]['h5_acquisition_date']; naip_date=GRANULE_AUDITS[stem]['naip_date']
  ax.set(title=f'Image-only ROI audit | {stem}\nNAIP {naip_date}; CASALS H5 {date_value}',xlabel=f'Easting (m, {ds.crs.to_string()})',ylabel='Northing (m)')
  handles,labels=ax.get_legend_handles_labels()
  if handles: ax.legend(handles,labels,loc='upper right',title='ROI tier')
  save_figure(fig,OUT_ROOT,f'naip_roi_board_{stem}.png')

# Track coverage retains every ROI's full pulse distribution.
roi_plot=ROI_AUDIT[ROI_AUDIT.n_nominal_pulse_centers>0].copy()
max_rows=max(1,int(roi_plot.groupby('granule_id').size().max()))
fig,axes=plt.subplots(1,2,figsize=(16,max(6,0.45*max_rows)),constrained_layout=True)
for ax,h5_path in zip(axes,H5_PATHS):
 stem=h5_path.stem; data=roi_plot[roi_plot.granule_id==stem].sort_values('n_tracks')
 colors=['#2ca58d' if t=='primary' else '#e0a537' for t in data.analysis_tier]
 ax.barh(data.roi_id,data.n_tracks,color=colors); ax.set(title=stem.replace('casals_l1b_',''),xlabel='Track count',ylabel='ROI')
 ax.grid(axis='x',alpha=.2)
fig.suptitle('ROI track coverage | primary and exploratory ROIs',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'roi_track_coverage.png')

# SNR distributions are shown within each granule; low-SNR pulses remain in the plots.
fig,axes=plt.subplots(1,2,figsize=(16,7),constrained_layout=True)
for ax,h5_path in zip(axes,H5_PATHS):
 stem=h5_path.stem; frame=ROI_PULSE_FEATURES[ROI_PULSE_FEATURES.granule_id==stem]
 rois=[rid for rid in frame.roi_id.unique()]
 arrays=[frame.loc[frame.roi_id==rid,'refh_snr'].dropna().to_numpy() for rid in rois]
 if arrays:
  ax.boxplot(arrays,showfliers=False,showmeans=True,meanline=True)
  ax.set_xticks(np.arange(1,len(rois)+1),rois,rotation=45,ha='right',fontsize=8)
 ax.set(title=stem.replace('casals_l1b_',''),ylabel='Native H5 Refh SNR',xlabel='ROI')
fig.suptitle('Within-granule Refh SNR distributions (all ROI pulses)',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'refh_snr_by_roi.png')

# Candidate-screen rate denominators include every ROI-center pulse.
fig,axes=plt.subplots(1,2,figsize=(16,7),constrained_layout=True)
for ax,h5_path in zip(axes,H5_PATHS):
 stem=h5_path.stem; data=ROI_RESPONSE[ROI_RESPONSE.granule_id==stem].sort_values('candidate_signal_rate_all_roi_pulses')
 colors=['#2ca58d' if t=='primary' else '#e0a537' for t in data.analysis_tier]
 ax.barh(data.roi_id,data.candidate_signal_rate_all_roi_pulses,color=colors)
 for y,row in enumerate(data.itertuples()): ax.text((row.candidate_signal_rate_all_roi_pulses or 0)+.005,y,f'n={row.n_roi_center_pulses:,}',va='center',fontsize=7)
 ax.set_xlim(0,max(.05,float(data.candidate_signal_rate_all_roi_pulses.max())*1.25)); ax.set(title=stem.replace('casals_l1b_',''),xlabel='Candidate signals / all ROI-center pulses',ylabel='ROI')
fig.suptitle('Noise-calibrated candidate-signal screen; not official return detection',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'noise_calibrated_candidate_rate.png')

# Paired common-track contrasts; each dot is one shared track and is not an independent site.
primary_matched=MATCHED_TRACK_COMPARISONS[(MATCHED_TRACK_COMPARISONS.analysis_tier=='primary_pair')&MATCHED_TRACK_COMPARISONS.metric.isin(['refh_snr','refh_amp_counts','candidate_signal_rate'])].copy()
primary_matched=primary_matched[primary_matched.apply(lambda row:int(row.track_num) in set(GLOBAL_COMMON_TRACKS.get(row.granule_id,[])),axis=1)]
pair_keys=primary_matched[['granule_id','roi_a','roi_b']].drop_duplicates(); pair_labels=[f"{r.granule_id.split('_')[2]} {r.roi_a} − {r.roi_b}" for r in pair_keys.itertuples()]
fig,axes=plt.subplots(1,3,figsize=(max(15,1.4*len(pair_keys)),7),constrained_layout=True)
for ax,metric,title in zip(axes,['refh_snr','refh_amp_counts','candidate_signal_rate'],['SNR median difference','Amplitude median difference (counts)','Candidate-rate difference']):
 subset=primary_matched[primary_matched.metric==metric]; labels=[]; groups=[]
 for _,pair in pair_keys.iterrows():
  values=subset[(subset.granule_id==pair.granule_id)&(subset.roi_a==pair.roi_a)&(subset.roi_b==pair.roi_b)].difference_a_minus_b.dropna().to_numpy()
  if values.size: labels.append(f"{pair.granule_id.split('_')[2]}\n{pair.roi_a} − {pair.roi_b}"); groups.append(values)
 if groups:
  ax.boxplot(groups,showfliers=False); rng=np.random.default_rng(20261012)
  for i,values in enumerate(groups,1): ax.scatter(rng.normal(i,0.035,len(values)),values,s=9,alpha=.6,color='#275d87')
  ax.set_xticks(np.arange(1,len(labels)+1),labels,rotation=60,ha='right',fontsize=7)
 ax.axhline(0,color='black',lw=1,ls='--'); ax.set(title=title,ylabel='ROI A − ROI B (within shared track)'); ax.grid(axis='y',alpha=.2)
fig.suptitle('Within-granule matched-track contrasts | descriptive, not site-level replication',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'matched_track_landcover_comparison.png')

# Width and energy are summarized only among noise-screen candidates.
fig,axes=plt.subplots(1,2,figsize=(16,7),constrained_layout=True)
for ax,metric,title,unit in zip(axes,['width_halfmax_bins_candidate_median','dominant_support_energy_counts_bin_candidate_median'],['Candidate peak width','Candidate dominant-support energy'],['bins','counts × bins']):
 data=ROI_RESPONSE[ROI_RESPONSE[metric].notna()].sort_values(metric)
 if len(data):
  colors=['#2ca58d' if t=='primary' else '#e0a537' for t in data.analysis_tier]
  ax.barh(data.roi_id,data[metric],color=colors)
  for y,row in enumerate(data.itertuples()):
   n=getattr(row,metric.replace('_median','_n')) if hasattr(row,metric.replace('_median','_n')) else 0
   ax.text(getattr(row,metric)+max(float(data[metric].max())*.01,0.01),y,f'n={n}',va='center',fontsize=7)
 ax.set(title=title,xlabel=unit,ylabel='ROI')
fig.suptitle('Waveform morphology among candidate signals only',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'candidate_width_energy_by_roi.png')

# Site-level points show within-class and between-site spread without pulse-level p-values.
fig,axes=plt.subplots(2,2,figsize=(15,10),constrained_layout=True)
for row,h5_path in enumerate(H5_PATHS):
 stem=h5_path.stem; data=SITE_RESPONSE[(SITE_RESPONSE.granule_id==stem)&(SITE_RESPONSE.analysis_tier=='primary')]
 classes=list(dict.fromkeys(data.class_label.tolist()))
 for col,metric,title in [(0,'refh_snr_median','Site median Refh SNR'),(1,'candidate_signal_rate_all_site_pulses','Site candidate-signal rate')]:
  ax=axes[row,col]
  for xpos,cls in enumerate(classes):
   group=data[data.class_label==cls]
   ax.scatter(np.full(len(group),xpos),group[metric],s=55)
   for item in group.itertuples(): ax.annotate(item.site_id,(xpos,getattr(item,metric)),xytext=(4,4),textcoords='offset points',fontsize=7)
  ax.set_xticks(np.arange(len(classes)),classes,rotation=25,ha='right'); ax.set(title=f'{stem.split("_")[2]} | {title}',ylabel=metric.replace('_',' '),xlabel='Image-context class'); ax.grid(axis='y',alpha=.2)
fig.suptitle('Independent unit is site; points show each primary site',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'site_level_variability.png')

# Six actual waveforms: one per granule and image-context class when available.
case_specs=[]
for h5_path in H5_PATHS:
 stem=h5_path.stem; frame=ROI_PULSE_FEATURES[ROI_PULSE_FEATURES.granule_id==stem]
 for cls in ['open_field','tree_canopy','road_surface']:
  roi_info=ROI_AUDIT[(ROI_AUDIT.granule_id==stem)&(ROI_AUDIT.class_label==cls)&(ROI_AUDIT.n_nominal_pulse_centers>0)]
  if roi_info.empty: continue
  roi_info=roi_info.sort_values(['analysis_tier','roi_id']); chosen_roi=roi_info.iloc[0].roi_id
  group=frame[frame.roi_id==chosen_roi]
  candidates=group[group.candidate_signal_detected]
  pool=candidates if len(candidates) else group[group.waveform_status=='no_candidate']
  if pool.empty: pool=group
  center=float(pool.refh_snr.median()); chosen=pool.iloc[int(np.argmin(np.abs(pool.refh_snr.to_numpy()-center)))]
  case_specs.append({'granule_id':stem,'class_label':cls,'roi_id':chosen_roi,'analysis_tier':roi_info.iloc[0].analysis_tier,'pulse_index':int(chosen.pulse_index),'selection_rule':'within ROI, candidate pool if available; choose the pulse nearest median SNR (no extreme-response selection)','candidate_signal_detected':bool(chosen.candidate_signal_detected),'refh_snr':float(chosen.refh_snr),'track_num':int(chosen.track_num),'sweep_num':int(chosen.sweep_num)})
fig,axes=plt.subplots(2,3,figsize=(16,9),constrained_layout=True); axes=axes.ravel(); waveform_manifest=[]
for ax,case in zip(axes,case_specs):
 h5_path=next(p for p in H5_PATHS if p.stem==case['granule_id'])
 with h5py.File(h5_path,'r') as h5:
  wave=np.asarray(h5['rx_waveform'][case['pulse_index'],:]); bg=float(h5['bg_mean'][case['pulse_index']])
 ax.plot(np.arange(len(wave)),wave,lw=.8,color='#254e70'); ax.axhline(bg,color='#777777',ls='--',lw=1,label='H5 bg_mean')
 if case['candidate_signal_detected']:
  row=ROI_PULSE_FEATURES[(ROI_PULSE_FEATURES.granule_id==case['granule_id'])&(ROI_PULSE_FEATURES.pulse_index==case['pulse_index'])].iloc[0]
  ax.axvline(int(row.candidate_peak_bin),color='#c74747',ls=':',lw=1.2,label='candidate peak')
 ax.set(title=f"{case['granule_id'].split('_')[2]} | {case['roi_id']}\npulse {case['pulse_index']} | track {case['track_num']} | sweep {case['sweep_num']} | SNR {case['refh_snr']:.2f} | {'candidate' if case['candidate_signal_detected'] else 'no candidate'}",xlabel='RX bin',ylabel='RX amplitude (counts)')
 ax.legend(fontsize=7,loc='upper right'); waveform_manifest.append(case)
for ax in axes[len(case_specs):]: ax.axis('off')
fig.suptitle('Representative real H5 RX waveforms; selected near within-pool median SNR',fontsize=15,fontweight='bold')
save_figure(fig,OUT_ROOT,'representative_waveform_gallery.png')
pd.DataFrame(waveform_manifest).to_csv(OUT_ROOT/'representative_waveform_manifest.csv',index=False)

FIGURE_OUTPUTS=['roi_track_coverage.png','refh_snr_by_roi.png','noise_calibrated_candidate_rate.png','matched_track_landcover_comparison.png','candidate_width_energy_by_roi.png','site_level_variability.png','representative_waveform_gallery.png']+[f"naip_roi_board_{p.stem}.png" for p in H5_PATHS]
print({'figures':FIGURE_OUTPUTS,'naip_roi_board_count':len(H5_PATHS),'matched_track_pairs':len(pair_keys),'representative_waveforms':len(waveform_manifest)})


{'granule': 'casals_l1b_20241112T165718_001_02', 'primary_roi_n': 3, 'all_primary_roi_common_track_n': 0, 'track_min': None, 'track_max': None}
{'granule': 'casals_l1b_20241118T171757_001_02', 'primary_roi_n': 6, 'all_primary_roi_common_track_n': 23, 'track_min': 56, 'track_max': 78}


{'figures': ['roi_track_coverage.png', 'refh_snr_by_roi.png', 'noise_calibrated_candidate_rate.png', 'matched_track_landcover_comparison.png', 'candidate_width_energy_by_roi.png', 'site_level_variability.png', 'representative_waveform_gallery.png', 'naip_roi_board_casals_l1b_20241112T165718_001_02.png', 'naip_roi_board_casals_l1b_20241118T171757_001_02.png'], 'naip_roi_board_count': 2, 'matched_track_pairs': 11, 'representative_waveforms': 6}


In [7]:
# Freeze output inventory and report current matched-track coverage from the actual ROI polygons.
COMMON_TRACK_COUNTS=[]
for stem in [p.stem for p in H5_PATHS]:
 track_rows=ROI_TRACK_SUMMARY[ROI_TRACK_SUMMARY.granule_id==stem]
 roi_ids=list(track_rows.roi_id.unique())
 by_id={rid:track_rows[track_rows.roi_id==rid].track_num.astype(int).tolist() for rid in roi_ids}
 audit_map=ROI_AUDIT[ROI_AUDIT.granule_id==stem].set_index('roi_id')
 for i,ra in enumerate(roi_ids):
  for rb in roi_ids[i+1:]:
   if audit_map.loc[ra,'class_label']==audit_map.loc[rb,'class_label']: continue
   shared=common_track_ids(by_id[ra],by_id[rb])
   COMMON_TRACK_COUNTS.append({'granule_id':stem,'roi_a':ra,'roi_b':rb,'class_a':audit_map.loc[ra,'class_label'],'class_b':audit_map.loc[rb,'class_label'],'common_track_count':len(shared),'common_track_min':min(shared) if shared else None,'common_track_max':max(shared) if shared else None,'comparability':'common tracks present' if shared else 'not comparable'})
COMMON_TRACK_COUNTS=pd.DataFrame(COMMON_TRACK_COUNTS)
COMMON_TRACK_COUNTS.to_csv(OUT_ROOT/'common_track_coverage.csv',index=False)

ALL_PRIMARY_PAIR_COMPARISONS=MATCHED_TRACK_COMPARISONS[MATCHED_TRACK_COMPARISONS.analysis_tier=='primary_pair'].copy()
ALL_PRIMARY_PAIR_COMPARISONS=ALL_PRIMARY_PAIR_COMPARISONS[ALL_PRIMARY_PAIR_COMPARISONS.apply(lambda row: pd.notna(row.track_num) and int(row.track_num) in set(GLOBAL_COMMON_TRACKS.get(row.granule_id,[])),axis=1)]
ALL_PRIMARY_PAIR_COMPARISONS['common_track_scope']='intersection across all Included primary ROIs in the granule'
ALL_PRIMARY_PAIR_COMPARISONS.to_csv(OUT_ROOT/'all_primary_roi_common_track_comparisons.csv',index=False)
ALL_PRIMARY_COMMON_TRACKS=pd.DataFrame([{'granule_id':stem,'primary_roi_n':len(GLOBAL_PRIMARY_ROIS[stem]),'common_track_n':len(ids),'common_track_min':min(ids) if ids else None,'common_track_max':max(ids) if ids else None,'common_track_ids':'|'.join(map(str,ids)),'scope':'intersection across all Included primary ROIs in this granule'} for stem,ids in GLOBAL_COMMON_TRACKS.items()])
ALL_PRIMARY_COMMON_TRACKS.to_csv(OUT_ROOT/'all_primary_roi_common_tracks.csv',index=False)
SUMMARY['all_primary_roi_common_tracks']=ALL_PRIMARY_COMMON_TRACKS.to_dict('records')
SUMMARY['all_primary_roi_common_track_comparison_rows']=int(len(ALL_PRIMARY_PAIR_COMPARISONS))
SUMMARY['matched_track_comparison_scope']='Pairwise intersections are available in common_track_coverage.csv; main matched-track figure/table use the conservative intersection across all Included primary ROIs within each granule.'


SUMMARY['common_track_counts']=COMMON_TRACK_COUNTS.to_dict('records')
SUMMARY['figure_outputs']=FIGURE_OUTPUTS
SUMMARY['output_files']=['all_primary_roi_common_tracks.csv','all_primary_roi_common_track_response_summary.csv','all_primary_roi_common_track_comparisons.csv','roi_audit.csv','noise_peak_calibration.json','noise_tail_stationarity.csv','roi_response_summary.csv','roi_track_response_summary.csv','roi_pulse_features.csv.gz (one per granule)','matched_track_comparisons.csv','matched_track_pair_summary.csv','common_track_coverage.csv','site_level_response_summary.csv','site_level_variability.csv','representative_waveform_manifest.csv','landcover_response_summary.json']+FIGURE_OUTPUTS
SUMMARY['statistical_independence_note']='Pulses and tracks are nested observations. Site/ROI identity is the primary independent unit; with few sites, all comparisons remain descriptive and no pulse-level significance tests are reported.'
(OUT_ROOT/'landcover_response_summary.json').write_text(json.dumps(json_safe_n11(SUMMARY),indent=2,allow_nan=False),encoding='utf-8')

print('Pairwise November 18 open-field vs canopy coverage:')
nov18=COMMON_TRACK_COUNTS[(COMMON_TRACK_COUNTS.granule_id=='casals_l1b_20241118T171757_001_02')&(COMMON_TRACK_COUNTS.roi_a=='open_field_01')&(COMMON_TRACK_COUNTS.roi_b=='tree_canopy_02')]
if nov18.empty:
 nov18=COMMON_TRACK_COUNTS[(COMMON_TRACK_COUNTS.granule_id=='casals_l1b_20241118T171757_001_02')&((COMMON_TRACK_COUNTS.roi_a=='open_field_01')|(COMMON_TRACK_COUNTS.roi_b=='open_field_01'))&((COMMON_TRACK_COUNTS.roi_a=='tree_canopy_02')|(COMMON_TRACK_COUNTS.roi_b=='tree_canopy_02'))]
print(nov18.to_dict('records'))
print('Conservative all-primary-ROI intersection:',ALL_PRIMARY_COMMON_TRACKS[ALL_PRIMARY_COMMON_TRACKS.granule_id.str.contains('20241118')].to_dict('records'))
print('Noise-null simulation control:')
display(pd.DataFrame([{'granule':k,'prominence_threshold_sigma':v['threshold_peak_prominence_sigma'],'synthetic_validation_false_alarm_rate':v['validation_false_alarm_rate'],'noise_windows':v['real_background_window_count'],'tail_sd_ratio':v['tail_segment_sd_ratio_max_min'],'detection_probability_calibrated':v['detection_probability_calibrated']} for k,v in NOISE_CALIBRATIONS.items()]))
print('Per-ROI all-pulse response summary:')
display(ROI_RESPONSE[['granule_id','roi_id','analysis_tier','n_roi_center_pulses','n_valid_waveforms','n_candidate_signals','candidate_signal_rate_all_roi_pulses','refh_snr_median','refh_amp_counts_median','width_halfmax_bins_candidate_n']])
print('Primary independent-site counts:')
print(json.dumps(SITE_UNIT_COUNTS,indent=2))
print('All figures are saved under',OUT_ROOT)


Pairwise November 18 open-field vs canopy coverage:
[{'granule_id': 'casals_l1b_20241118T171757_001_02', 'roi_a': 'open_field_01', 'roi_b': 'tree_canopy_02', 'class_a': 'open_field', 'class_b': 'tree_canopy', 'common_track_count': 55, 'common_track_min': 56.0, 'common_track_max': 110.0, 'comparability': 'common tracks present'}]
Conservative all-primary-ROI intersection: [{'granule_id': 'casals_l1b_20241118T171757_001_02', 'primary_roi_n': 6, 'common_track_n': 23, 'common_track_min': 56.0, 'common_track_max': 78.0, 'common_track_ids': '56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78', 'scope': 'intersection across all Included primary ROIs in this granule'}]
Noise-null simulation control:


,granule,prominence_threshold_sigma,synthetic_validation_false_alarm_rate,noise_windows,tail_sd_ratio,detection_probability_calibrated
0,casals_l1b_20241112T165718_001_02,10.884768,0.0058,6144,1.009856,False
1,casals_l1b_20241118T171757_001_02,10.194930,0.0029,6144,1.009591,False


Per-ROI all-pulse response summary:


,granule_id,roi_id,analysis_tier,n_roi_center_pulses,n_valid_waveforms,n_candidate_signals,candidate_signal_rate_all_roi_pulses,refh_snr_median,refh_amp_counts_median,width_halfmax_bins_candidate_n
0,casals_l1b_20241112T165718_001_02,open_field_12_01,primary,17974,17974,5223,0.290586,2.379448,656.0,5223
1,casals_l1b_20241112T165718_001_02,open_field_12_02,primary,17171,17171,4374,0.254732,2.264305,665.0,4374
2,casals_l1b_20241112T165718_001_02,road_core_12_01,exploratory,3820,3820,1026,0.268586,2.246030,588.0,1026
3,casals_l1b_20241112T165718_001_02,tree_canopy_12_01,primary,32014,32014,1283,0.040076,1.911554,495.0,1283
4,casals_l1b_20241112T165718_001_02,water_pond_12_01,exploratory,10160,10160,572,0.056299,1.947419,479.0,572
5,casals_l1b_20241118T171757_001_02,open_field_01,primary,7140,7140,7014,0.982353,7.121695,1799.0,7014
6,casals_l1b_20241118T171757_001_02,open_field_02,primary,8924,8924,8663,0.970753,6.074582,1992.0,8663
7,casals_l1b_20241118T171757_001_02,road_core_01,primary,1040,1040,955,0.918269,6.336636,1690.0,955
8,casals_l1b_20241118T171757_001_02,tree_canopy_01,primary,3503,3503,2586,0.738224,3.494794,864.0,2586
9,casals_l1b_20241118T171757_001_02,tree_canopy_02,primary,4329,4329,2824,0.652345,3.250560,864.0,2824


Primary independent-site counts:
{
  "casals_l1b_20241112T165718_001_02": {
    "open_field": {
      "n_independent_sites": 2,
      "site_ids": [
        "nov12_east_cropland_01",
        "nov12_southeast_cropland_02"
      ]
    },
    "tree_canopy": {
      "n_independent_sites": 1,
      "site_ids": [
        "nov12_south_forest_stand_01"
      ]
    }
  },
  "casals_l1b_20241118T171757_001_02": {
    "open_field": {
      "n_independent_sites": 2,
      "site_ids": [
        "field_site_01",
        "field_site_02"
      ]
    },
    "road_surface": {
      "n_independent_sites": 1,
      "site_ids": [
        "farm_road_segment_01"
      ]
    },
    "tree_canopy": {
      "n_independent_sites": 2,
      "site_ids": [
        "nov18_northwest_woodlot_01",
        "south_central_forest_patch"
      ]
    }
  }
}
All figures are saved under C:\Users\zhong267\OneDrive - purdue.edu\CASALS\outputs\notebooks\11_landcover_lidar_response\revised


## Interpretation boundaries

Results are stratified by granule and ROI. Track-matched differences control track identity within each granule, while each ROI-track uses all pulses and reports its sample size, percentiles, and candidate-signal rate. Tracks and pulses remain nested observations; the independent units are sites. The two adjacent November 18 canopy polygons share one site ID and count as one forest site.

The tail-window block bootstrap controls the multiple search over 2,728 RX bins under an exchangeable-noise assumption. The recorded segment diagnostics show how similar the sampled tail windows are. The threshold is a provisional candidate screen: simulated null false alarms do not establish detection probability for real returns. Width, support energy, and asymmetry are only reported for threshold-passing candidates.

`refh_amp_counts` is native digital amplitude. This notebook does not estimate wavelength-resolved spectral reflectance, calibrate radiometry, or infer causal land-cover effects.